# Polars Expressions API: Selection, Filtering, Aggregation, and Joins

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

The heart of Polars is its **Expression API**. Rather than mutating arrays in-place or executing eager line-by-line Python functions, Polars expressions represent modular mathematical functions that take an Arrow Series and return an Arrow Series:

$$\text{Expression}: \text{Series} \to \text{Series}$$

Expressions can be combined, parallelized, and optimized into an abstract execution graph prior to computation.

### Learning Objectives
1. Master column projections with `pl.col()`, `pl.all()`, and `pl.exclude()`.
2. Construct complex boolean filter predicates using bitwise logic (`&`, `|`, `~`).
3. Add and derive feature columns with `with_columns()` and `.alias()`.
4. Perform multi-metric group-by aggregations with `group_by()` and `.agg()`.
5. Execute relational table joins (`inner`, `left`, `right`) across structured schemas.


In [ ]:
from pathlib import Path

import polars as pl

DATA_DIR = Path("..") / "data"
FOOD_PATH = DATA_DIR / "food_products.csv"
CAT_PATH = DATA_DIR / "categories_metadata.csv"

food_df = pl.read_csv(FOOD_PATH)
cat_df = pl.read_csv(CAT_PATH)

print("Food Products DataFrame:")
print(food_df.head(4))
print("Categories Metadata DataFrame:")
print(cat_df.head(4))

## 1. Column Selection and Projections

In Polars, column selection is performed via `select()`. Within `select()`, we express column targets using `pl.col()`.


In [ ]:
# 1. Select specific columns
print("Select product_name and category:")
print(food_df.select(["product_name", "category"]).head(3))

# 2. Select using column expressions with transformations
print("Select with expression-level transformation:")
print(
    food_df.select(
        [pl.col("product_name"), (pl.col("energy_kcal") * 4.184).round(1).alias("energy_kj")]
    ).head(3)
)

# 3. Select all except specific columns
print("Exclude metadata columns:")
print(food_df.select(pl.exclude(["created_date", "brand"])).head(3))

## 2. Predicate Filtering: Scalar Conditions and Ranges

The `filter()` method accepts boolean expressions to subset rows. Multiple predicates are combined using bitwise operators:
- `&` : Logical AND
- `|` : Logical OR
- `~` : Logical NOT


In [ ]:
# Filter low calorie and high protein products
healthy_df = food_df.filter((pl.col("energy_kcal") < 200) & (pl.col("protein_g") >= 5.0))
print(f"Products matching (energy < 200 kcal AND protein >= 5g): {healthy_df.height}")
print(healthy_df.select(["product_name", "category", "energy_kcal", "protein_g"]))

# Range filtering with is_between
mid_fat_df = food_df.filter(pl.col("fat_g").is_between(5.0, 30.0))
print(f"Products with fat between 5g and 30g: {mid_fat_df.height}")
print(mid_fat_df.select(["product_name", "fat_g"]).head(3))

## 3. Deriving Columns with `with_columns`

The `with_columns()` method evaluates expressions and appends or replaces columns while keeping existing columns intact.


In [ ]:
# Calculate total macronutrient weight and caloric breakdown
enriched_df = food_df.with_columns(
    [
        (pl.col("fat_g") + pl.col("carbohydrates_g") + pl.col("protein_g"))
        .round(2)
        .alias("total_macros_g"),
        ((pl.col("protein_g") * 4) / pl.col("energy_kcal").clip(lower_bound=1) * 100)
        .round(1)
        .alias("protein_cal_pct"),
    ]
)

print("Enriched DataFrame with Derived Macro Columns:")
print(enriched_df.select(["product_name", "total_macros_g", "protein_cal_pct"]).head(5))

## 4. Grouping and Aggregations with `group_by` and `.agg`

Polars provides high-speed multi-threaded group-by operations. We pass a list of aggregation expressions to `.agg()`.


In [ ]:
# Compute multi-metric aggregations per category
category_summary = (
    food_df.group_by("category", maintain_order=True)
    .agg(
        [
            pl.len().alias("product_count"),
            pl.col("energy_kcal").mean().round(1).alias("mean_kcal"),
            pl.col("protein_g").mean().round(2).alias("mean_protein"),
            pl.col("fat_g").max().alias("max_fat"),
        ]
    )
    .sort("product_count", descending=True)
)

print("Category Summary Aggregations:")
print(category_summary)

## 5. Relational Table Joins

We join `food_products.csv` with `categories_metadata.csv` using `join()`. Supported join strategies include `inner`, `left`, `right`, and `full`.


In [ ]:
# Left join products with category metadata
merged_df = food_df.join(cat_df, on="category", how="left")

print(f"Joined DataFrame shape: {merged_df.shape}")
print(
    merged_df.select(["product_name", "category", "department", "storage_type", "tax_rate"]).head(6)
)

# Verify join consistency
assert merged_df.height == food_df.height
print("Join completed with 100% record retention.")

## 6. Practice Challenge: End-to-End Analytics Pipeline

**Exercise:** Build a complete data transformation pipeline:
1. Join `food_df` with `cat_df`.
2. Filter for items stored in `Ambient` or `Dry Storage`.
3. Compute the effective tax amount per unit assuming an arbitrary base retail price ($P = \text{energy\_kcal} \times 0.02$).
4. Group by `department` and aggregate the total estimated tax revenue and average protein content.


In [ ]:
pipeline_result = (
    food_df.join(cat_df, on="category", how="inner")
    .filter(pl.col("storage_type").is_in(["Ambient", "Dry Storage"]))
    .with_columns(
        [
            (pl.col("energy_kcal") * 0.02).round(2).alias("base_price"),
            ((pl.col("energy_kcal") * 0.02) * pl.col("tax_rate")).round(2).alias("tax_amount"),
        ]
    )
    .group_by("department", maintain_order=True)
    .agg(
        [
            pl.len().alias("item_count"),
            pl.col("tax_amount").sum().round(2).alias("total_tax"),
            pl.col("protein_g").mean().round(1).alias("avg_protein_g"),
        ]
    )
    .sort("item_count", descending=True)
)

print("Pipeline Aggregation Summary:")
print(pipeline_result)

assert pipeline_result.height > 0
print("Challenge assertions successfully verified!")